<!-- SPDX-FileCopyrightText: 2026 NVIDIA CORPORATION & AFFILIATES -->
<!-- SPDX-License-Identifier: Apache-2.0 AND CC-BY-4.0 -->

<br>
<a href="https://www.nvidia.com/en-us/training/">
    <div style="width: 55%; background-color: white; margin-top: 50px;">
    <img src="../images/nvidia.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px"/></div>
</a>
<h1 style="line-height: 1.4;"><font color="#76b900"><b>Building LLM Applications With Prompt Engineering</b></font></h1>
<h2><b>Streaming and Batching</b></h2>
<br>

Code cells in this notebook are licensed under
[Apache-2.0](../../../LICENSES/Apache-2.0.txt). Markdown and raw cells are licensed
under [CC-BY-4.0](../../../LICENSES/CC-BY-4.0.txt). See the
[combined license terms](../../../LICENSES/CC-BY-4.0-Apache2_Dual_License.txt).
Third-party material retains its original license and is identified in
[`THIRD_PARTY_NOTICES.md`](../../../THIRD_PARTY_NOTICES.md).

Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES.


In this notebook you'll learn how to `stream` model responses and handle multiple chat completion requests in batches.

---

## Objectives

By the time you complete this notebook, you will:

- Learn to `stream` model responses.
- Learn to batch model responses.
- Compare the performance of batch processing to single prompt chat completion.

---

## Imports

Here we import the `ChatNVIDIA` class from `langchain_nvidia_ai_endpoints`, which will enable us to interact with the remote `nvidia/nemotron-3-super-120b-a12b` endpoint.

In [ ]:
import os
from langchain_nvidia_ai_endpoints import ChatNVIDIA

---

## Create a Model Instance

### API key for this notebook kernel

Paste the API key for your configured endpoint when the next cell prompts for it. The typed value is visible during entry but is not placed in the cell source or written to a file. It remains only in this kernel's memory. Restarting the kernel discards it, and a separate notebook kernel will ask for the key again.

Avoid entering the key while screen sharing or recording.

In [ ]:
import os

if not os.environ.get("NVIDIA_API_KEY"):
    _api_key = input("Paste the API key for the configured endpoint: ").strip()
    if not _api_key:
        raise ValueError("An API key is required to run this notebook.")
    os.environ["NVIDIA_API_KEY"] = _api_key
    del _api_key

os.environ.setdefault("NVIDIA_BASE_URL", "https://integrate.api.nvidia.com/v1")

# To use an alternative endpoint, uncomment below and change the URL.
# os.environ["NVIDIA_BASE_URL"] = "https://your-endpoint.example.com/v1"
print("API access is configured for this notebook kernel.")

In [ ]:
base_url = os.getenv("NVIDIA_BASE_URL")
model = 'nvidia/nemotron-3-super-120b-a12b'
llm = ChatNVIDIA(
    base_url=base_url, model=model, temperature=0, max_completion_tokens=2048,
    model_kwargs={"chat_template_kwargs": {"enable_thinking": False}},
)

---

## Sanity Check

Before proceeding with new use cases, let's sanity check that we can interact with our course model endpoint via LangChain.

In [ ]:
prompt = 'Return only the founding year stated in this text: NVIDIA was founded in 1993.'
result = llm.invoke(prompt)

In [ ]:
print(result.content)

---

## Streaming Responses

As an alternative to the `invoke` method, you can use the `stream` method to receive the model response in chunks. This way, you don't have to wait for the entire response to be generated, and you can see the output as it is being produced. Especially for long responses, or in user-facing applications, streaming output can result in a much better user experience.

Let's create a prompt that generates a longer response.

In [ ]:
prompt = 'Explain how parallel computing works in roughly 500 words.'

Given this prompt, let's see how the `stream` function works.

In [ ]:
for chunk in llm.stream(prompt):
    print(chunk.content, end='')

The `stream` method in LangChain serves as a foundational tool and shows the response as it is being generated. This can make the interaction with the LLMs feel more responsive and improve the user experience.

---

Later notebooks use the same streaming interface to display responses as they arrive.

---

## Batching Responses

You can also use `batch` to call the prompts on a list of inputs. Calling `batch` will return a list of responses in the same order as they were passed in.

`batch` can schedule prompts concurrently when endpoint capacity allows. This course sets `max_concurrency=1` because the shared endpoint has a narrow request budget. The interface and output ordering stay the same, while requests run serially.

We'll demonstrate the batch interface and ordered results with this list of prompts about state capitals.

In [ ]:
state_capital_questions = [
    'What is the capital of California?',
    'What is the capital of Texas?',
    'What is the capital of New York?',
    'What is the capital of Florida?',
    'What is the capital of Illinois?',
    'What is the capital of Ohio?'
]

Using `batch` we can pass in the entire list...

In [ ]:
capitals = llm.batch(state_capital_questions, config={"max_concurrency": 1})

... and get back a list of responses.

In [ ]:
len(capitals)

In [ ]:
for capital in capitals:
    print(capital.content)

One thing to note is that `batch` is not engaging with the LLM in a multi-turn conversation (a topic we will cover at length later in the workshop). It sends each prompt as an independent request and returns the responses in input order.

---

## Comparing batch and `invoke`

Here we time one serialized `batch` call. In this course runtime, it demonstrates the collection interface rather than a concurrency speedup. Note the `Wall time`.

In [ ]:
%%time
llm.batch(state_capital_questions, config={"max_concurrency": 1})

For comparison, iterate over `state_capital_questions` and call `invoke` on each item. Both paths are serialized here. Timings include client, transport and replay overhead; these measurements do not establish parallel inference speedup.

In [ ]:
%%time
for cq in state_capital_questions:
    llm.invoke(cq)

---

## Exercise: Batch Process to Create an FAQ Document

For this exercise you'll use batch processing to respond to a variety of LLM-related questions in service of creating an FAQ document (in this notebook setting the document will just be something we print to screen).

Here is a list of LLM-related questions.

In [ ]:
faq_questions = [
    'What is a Large Language Model (LLM)?',
    'How do LLMs work?',
    'What are some common applications of LLMs?',
    'What is fine-tuning in the context of LLMs?',
    'How do LLMs handle context?',
    'What are some limitations of LLMs?',
    'How do LLMs generate text?',
    'What is the importance of prompt engineering in LLMs?',
    'How can LLMs be used in chatbots?',
    'What are some ethical considerations when using LLMs?'
]

Your job is to populate `faq_answers` with one complete, brief answer per question. Ask for at most two sentences per answer and use `batch` to preserve the question order.

Upon successful completion, you should be able to print the return value of calling the following `create_faq_document` with `faq_questions` and `faq_answers` and get an FAQ document for all of the LLM-related questions above.

In [ ]:
def create_faq_document(faq_questions, faq_answers):
    faq_document = ''
    for question, response in zip(faq_questions, faq_answers):
        faq_document += f'{question.upper()}\n\n'
        faq_document += f'{response.content}\n\n'
        faq_document += '-'*30 + '\n\n'

    return faq_document

If you get stuck, check out the *Solution* below.

### Your Work Here

In [ ]:
faq_answers = []

In [ ]:
# This should work after you successfully populate `faq_answers` with LLM responses.
print(create_faq_document(faq_questions, faq_answers))

### Solution

Ask for two brief sentences per question and leave enough token budget to finish each answer. The batch runs one request at a time; the original questions still become the FAQ headings.

In [ ]:
faq_answers = llm.bind(max_completion_tokens=512).batch(
    [
        f"Answer this FAQ question in at most two brief sentences: {question}"
        for question in faq_questions
    ],
    config={'max_concurrency': 1},
)

In [ ]:
def create_faq_document(faq_questions, faq_answers):
    faq_document = ''
    for question, response in zip(faq_questions, faq_answers):
        faq_document += f'{question.upper()}\n\n'
        faq_document += f'{response.content}\n\n'
        faq_document += '-'*30 + '\n\n'

    return faq_document
###################################################################
# Fancier solution: Create the markdown, and then render it
from IPython.display import display, Markdown

def md_blocq(body):
    return f"<blockquote>\n\n{body}\n\n</blockquote>"

def md_details(head, body):
    return f"<details><summary><b>{head}</b></summary>{md_blocq(body)}</details>"

def create_faq_document(faq_questions, faq_answers):
    faq_document = ''
    for question, response in zip(faq_questions, faq_answers):
        faq_document += md_details(question, response.content)

    return faq_document

Markdown(create_faq_document(faq_questions, faq_answers))

In [ ]:
print(create_faq_document(faq_questions, faq_answers))

---

## Summary

In this notebook you learned how to `stream` and batch model responses, and used batched LLM calls to generate a helpful FAQ document.

In the next notebook you'll begin focusing more heavily on the creation of prompts themselves with an emphasis on iterative prompt development and engineering prompts that are very specific.

---

<!-- NEXT_STEP_CARD -->

<div style="border-left: 6px solid #76B900; background: #f7fdf2; padding: 14px 18px; border-radius: 10px; margin: 20px 0;">
<p style="margin: 0 0 6px; color: #315c00; font-weight: 700; letter-spacing: .04em; text-transform: uppercase;">Continue the unified course path</p>
<p style="margin: 0 0 8px; color: black;"><strong>Next step:</strong> Open <code>1-Intro-to-Prompting/15-Iterative-Prompting.ipynb</code> next: <strong>Iterative Prompt Development</strong>.</p>
<p style="margin: 0; color: black;"><strong>Before moving on:</strong> Keep one case where streaming or batching changed the workflow payoff, so the next notebook can focus on improving the prompt itself rather than the transport pattern.</p>
</div>